* The original example asked users to set `DB_URI` to a database. This version uses deterministic local data, so neither `DB_URI` nor a database is required.

In [ ]:
import asyncio
import datetime as dt
import numpy as np


In [ ]:
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6


In [ ]:
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.portfolio.evolver import evolve_positions
from alphamind.portfolio.longshortbulder import long_short_builder


In [ ]:
from alphamind.settlement.simplesettle import simple_settle

latest = panel.filter(pc.equal(panel["trade_date"], dates[-1]))
signal = latest["factor"].to_numpy(zero_copy_only=False)


In [ ]:
weights = long_short_builder(signal, leverage=1.0).flatten()
returns = latest["dx"].to_numpy(zero_copy_only=False)
settlement = await asyncio.to_thread(
    simple_settle, weights, returns, groups=latest["industry"].to_pylist()
)
evolved = evolve_positions(weights, returns)


In [ ]:
assert abs(np.abs(weights).sum() - 1.0) < 1e-10
assert abs(np.abs(evolved).sum() - 1.0) < 1e-10
settlement
